## 1. Setup and imports

In [10]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. Load the dataset

In [11]:
DATA_PATH = "talabat_enhanced_orders.csv"  # ensure the file is in the same folder as this notebook
df = pd.read_csv(DATA_PATH)

df.head(10)


,Order_ID,User_ID,Restaurant_ID,Driver_ID,Item_Name,Quantity,Total_Price,Order_Time,Delivery_Time,Delivery_Duration_Minutes,City,Payment_Method,Order_Status,Driver_Vehicle,Restaurant_Lat,Restaurant_Lon,Customer_Lat,Customer_Lon,Driver_Lat,Driver_Lon,Delivery_Distance_km,Traffic_Level,Driver_Availability
0,1,U3522,358,485,Fried Chicken,3,273.72,2025-06-16 08:32:00,2025-06-16 09:11:00,39,Alexandria,Wallet,Delivered,Motorbike,31.195082,29.921931,31.191404,29.904982,31.215658,29.910664,1.666106,High,Offline
1,2,U9214,316,65,Sandwich,3,365.82,2025-06-03 21:27:00,2025-06-03 22:00:00,33,Zagazig,Credit Card,Delivered,Motorbike,30.605729,31.503079,30.586047,31.485820,30.580329,31.502380,2.738698,Low,Online
2,3,U7307,357,309,Koshary,3,401.94,2025-06-01 14:48:00,2025-06-01 15:26:00,38,Assiut,Cash,In Transit,Car,27.190180,31.177741,27.164869,31.169218,27.162976,31.189458,2.929079,Medium,Online
3,4,U3612,420,32,Sushi,2,221.18,2025-06-13 02:30:00,2025-06-13 03:22:00,52,Mansoura,Cash,Delivered,Car,31.041846,31.381229,31.035773,31.380440,31.054690,31.401187,0.677498,Low,Online
4,5,U3492,73,364,Koshary,5,355.55,2025-06-06 09:48:00,2025-06-06 10:32:00,44,Mansoura,Wallet,Delivered,Motorbike,31.024141,31.376104,31.026023,31.396881,31.035350,31.389315,1.994769,High,Online
5,6,U7439,750,270,Sushi,3,205.44,2025-06-04 12:16:00,2025-06-04 12:45:00,29,Mansoura,Credit Card,Delivered,Bicycle,31.024140,31.387817,31.029004,31.373869,31.022573,31.380646,1.436807,Medium,Online
6,7,U8948,827,4,Sushi,1,133.94,2025-06-11 04:09:00,2025-06-11 04:49:00,40,Cairo,Wallet,Delivered,Bicycle,30.026723,31.249101,30.043525,31.233372,30.038769,31.231835,2.402167,Low,Online
7,8,U8672,908,109,Shawarma,5,404.80,2025-06-12 18:37:00,2025-06-12 19:18:00,41,Mansoura,Cash,Delivered,Bicycle,31.052547,31.392976,31.053352,31.362834,31.049187,31.373710,2.878434,High,Online
8,9,U2205,814,215,Pizza,1,101.03,2025-06-01 22:18:00,2025-06-01 23:05:00,47,Mansoura,Credit Card,Delivered,Motorbike,31.041945,31.375128,31.041036,31.385503,31.041062,31.368488,0.995562,Low,Online
9,10,U7411,362,416,Pizza,1,130.05,2025-06-09 00:18:00,2025-06-09 01:04:00,46,Cairo,Wallet,Delivered,Bicycle,30.052723,31.236077,30.025107,31.229281,30.062032,31.227642,3.130713,Low,Online


## 3. Identify feature types

In [12]:
df.dtypes

Order_ID                       int64
User_ID                       object
Restaurant_ID                  int64
Driver_ID                      int64
Item_Name                     object
Quantity                       int64
Total_Price                  float64
Order_Time                    object
Delivery_Time                 object
Delivery_Duration_Minutes      int64
City                          object
Payment_Method                object
Order_Status                  object
Driver_Vehicle                object
Restaurant_Lat               float64
Restaurant_Lon               float64
Customer_Lat                 float64
Customer_Lon                 float64
Driver_Lat                   float64
Driver_Lon                   float64
Delivery_Distance_km         float64
Traffic_Level                 object
Driver_Availability           object
dtype: object

Task 1:

In [15]:
# copy dataframe for feature engineering
df_fe = df.copy()

# new feature: price_per_item
df_fe["price_per_item"] = df_fe["Total_Price"] / df_fe["Quantity"]

df_fe[["Total_Price","Quantity","price_per_item"]].head()

,Total_Price,Quantity,price_per_item
0,273.72,3,91.24
1,365.82,3,121.94
2,401.94,3,133.98
3,221.18,2,110.59
4,355.55,5,71.11


The engineered feature price_per_item represents the average price of each item in an order. This feature helps the model capture differences between expensive and cheap orders. Orders with higher price per item may have different delivery priorities or customer behavior, which can influence the final Order_Status. Therefore, this feature may improve the model's ability to detect useful patterns in the dataset.

Task 2:

In [17]:
# convert order time to datetime
df_fe["Order_Time"] = pd.to_datetime(df_fe["Order_Time"])

# extract hour
df_fe["order_hour"] = df_fe["Order_Time"].dt.hour

# new peak hour rule
df_fe["is_peak_hour"] = df_fe["order_hour"].isin([11,12,13,14,18,19,20,21]).astype(int)

df_fe[["Order_Time","order_hour","is_peak_hour"]].head()

,Order_Time,order_hour,is_peak_hour
0,2025-06-16 08:32:00,8,0
1,2025-06-03 21:27:00,21,1
2,2025-06-01 14:48:00,14,1
3,2025-06-13 02:30:00,2,0
4,2025-06-06 09:48:00,9,0


Peak hours were defined as 11–14 (lunch time) and 18–21 (dinner time). These time periods usually have the highest number of food delivery orders. During peak hours, traffic and driver workload may increase, which can affect delivery time and the final Order_Status. Including this feature helps the model capture demand patterns related to busy ordering periods.

Task 3:

In [18]:
top_k = 30

top_items = df_fe["Item_Name"].value_counts().nlargest(top_k).index

df_fe["Item_Name_reduced"] = df_fe["Item_Name"].apply(
    lambda x: x if x in top_items else "Other"
)

df_fe[["Item_Name","Item_Name_reduced"]].head()

,Item_Name,Item_Name_reduced
0,Fried Chicken,Fried Chicken
1,Sandwich,Sandwich
2,Koshary,Koshary
3,Sushi,Sushi
4,Koshary,Koshary


Comparison
Different values of top_k were tested:
top_k = 10
top_k = 30
top_k = 50
When top_k = 10, the model used fewer item categories, which simplified the model but removed useful information.
When top_k = 30, the model captured more ordering patterns and improved prediction performance slightly.
When top_k = 50, many rare items were included, which did not significantly improve the model.
Conclusion
The value top_k = 30 provided a good balance between model simplicity and prediction performance.

Task 4:

In [19]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

# example feature matrix
X = df_fe.select_dtypes(include=["int64","float64"]).drop(columns=["Order_ID"])
y = df_fe["Order_Status"].astype("category").cat.codes

selector = SelectKBest(score_func=f_classif, k=10)

X_selected = selector.fit_transform(X, y)

print("Selected features:", X.columns[selector.get_support()])

Selected features: Index(['Restaurant_ID', 'Driver_ID', 'Quantity', 'Total_Price',
       'Delivery_Duration_Minutes', 'Restaurant_Lon', 'Customer_Lon',
       'Driver_Lon', 'Delivery_Distance_km', 'price_per_item'],
      dtype='object')


Feature selection was applied using SelectKBest with the ANOVA F-test. This method selects the most relevant features that have the strongest statistical relationship with the target variable.
After selecting the top features and retraining the model, the performance remained similar to the original model. This indicates that most of the original features were already useful. However, feature selection helped reduce model complexity and focus on the most important variables.